# MOT baseline on Google Colab (GPU)

Этот ноутбук запускает `scripts/run_baseline.py` на GPU в Colab.

1. В Colab включи GPU: **Runtime -> Change runtime type -> T4/A100 GPU**
2. Обнови `REPO_URL`, `DATA_ROOT`, `SEQUENCES` в ячейке `Paths and run config`.
3. Запусти ячейки сверху вниз.

In [ ]:
!nvidia-smi

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/<your-user>/<your-repo>.git"
REPO_DIR = Path("/content/MOT")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

%cd /content/MOT
print('Python:', sys.version.split()[0])

In [ ]:
bash scripts/download_mot17_colab.sh

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install deps (оставляем torch из Colab, чтобы не ломать CUDA сборку)
!python -m pip install -U pip
!python -m pip install numpy pandas PyYAML motmetrics opencv-python ultralytics

In [ ]:
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
import yaml
from pathlib import Path

# Обнови под свои пути в Google Drive
DATA_ROOT = Path('/content/drive/MyDrive/datasets/MOT17/train')
SEQUENCES = ['MOT17-02-FRCNN', 'MOT17-04-FRCNN']
OUTDIR = Path('/content/MOT/baseline_runs/mot17_botsort_colab')

assert DATA_ROOT.exists(), f'Missing DATA_ROOT: {DATA_ROOT}'
for seq in SEQUENCES:
    seq_dir = DATA_ROOT / seq
    assert (seq_dir / 'img1').exists(), f'Missing frames: {seq_dir / "img1"}'
    assert (seq_dir / 'gt' / 'gt.txt').exists(), f'Missing GT: {seq_dir / "gt/gt.txt"}'

run_cfg = {
    'data': str(DATA_ROOT),
    'sequences': SEQUENCES,
    'outdir': str(OUTDIR),
    'device': '0',
    'model': 'yolo11n.pt',
    'conf': 0.25,
    'iou': 0.7,
    'imgsz': 1280,
    'classes': [0],
    'tracker_config': '/content/MOT/configs/botsort_baseline.yaml',
    'visualization': {
        'enabled': False,
        'fps': 30
    }
}

cfg_path = Path('/content/MOT/configs/run_baseline_colab.yaml')
with cfg_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(run_cfg, f, sort_keys=False, allow_unicode=False)

print('Config written to:', cfg_path)
print(cfg_path.read_text(encoding='utf-8'))

In [ ]:
%cd /content/MOT
!python scripts/run_baseline.py --config configs/run_baseline_colab.yaml

In [ ]:
from pathlib import Path

OUTDIR = Path('/content/MOT/baseline_runs/mot17_botsort_colab')
print('Metrics file:', OUTDIR / 'metrics.csv')
print((OUTDIR / 'metrics.csv').read_text(encoding='utf-8'))
print('Timing file:', OUTDIR / 'timing.csv')
print((OUTDIR / 'timing.csv').read_text(encoding='utf-8'))

In [ ]:
# Optional: сохранить результаты в Google Drive
from pathlib import Path

OUTDIR = Path('/content/MOT/baseline_runs/mot17_botsort_colab')
DRIVE_EXPORT = Path('/content/drive/MyDrive/MOT_runs/mot17_botsort_colab')
DRIVE_EXPORT.parent.mkdir(parents=True, exist_ok=True)
!cp -r {OUTDIR} {DRIVE_EXPORT}
print('Copied to:', DRIVE_EXPORT)